<a href="https://colab.research.google.com/github/Kiris-02/vox-persona/blob/main/vox_gpt_sovits_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ VOX IMPERIUM — GPT-SoVITS Cloud GPU Voice Clone Engine

**Mượn miễn phí card đồ họa NVIDIA T4 (16GB VRAM) của Google** để chạy GPT-SoVITS từ GitHub, clone giọng thật 100% cho 6 nhân vật:
- **Steve Jobs, Donald Trump, Elon Musk, Mark Zuckerberg, Nikola Tesla, Xi Jinping.**

👉 **Hướng dẫn chạy (Chỉ 1 thao tác duy nhất):**
1. Trên thanh menu, bấm **Runtime** (Thời gian chạy) -> chọn **Run all** (Chạy tất cả) hoặc bấm phím tắt `Ctrl + F9`.
2. Chờ hệ thống cài đặt và tải mô hình (khoảng 3-5 phút).
3. Ở ô code cuối cùng, copy đường link `https://xxx.trycloudflare.com` và dán vào mục Cài đặt trên web `vox-persona` là xong!

In [1]:
# 1. Kiểm tra card đồ họa NVIDIA T4
!nvidia-smi

Thu Sep 10 04:21:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Cấu hình kịch bản cài đặt GPT-SoVITS
%%writefile /content/setup.sh
set -e
cd /content

if [ ! -d "GPT-SoVITS" ]; then
    git clone https://github.com/RVC-Boss/GPT-SoVITS.git
fi

cd GPT-SoVITS

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS
pip install -q ipykernel pycloudflared fastapi uvicorn requests
bash install.sh --device CU126 --source HF --download-uvr5

Writing /content/setup.sh


In [ ]:
# 3. Tải Miniconda và tiến hành cài đặt GPT-SoVITS
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
!cd /content && bash setup.sh


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:01:58
🔁 Restarting kernel...
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5974, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 5974 (delta 3), reused 0 (delta 0), pack-reused 5954 (from 3)
Receiving objects: 100% (5974/5974), 14.27 MiB | 12.75 MiB/s, done.

In [ ]:
# 4. Tải các file mẫu âm thanh giọng thật của 6 nhân vật từ GitHub
import os, urllib.request

os.makedirs("/content/GPT-SoVITS/refs", exist_ok=True)
base_url = "https://raw.githubusercontent.com/Kiris-02/vox-persona/main/assets/audio/"
files = [
    "jobs_speech.mp3",
    "trump_speech.mp3",
    "musk_speech.wav",
    "zuck_speech.mp3",
    "tesla_speech.mp3",
    "xijinping_speech.ogg"
]
for f in files:
    dest = f"/content/GPT-SoVITS/refs/{f}"
    if not os.path.exists(dest):
        print(f"Downloading reference audio: {f}...")
        urllib.request.urlretrieve(base_url + f, dest)
print("✓ Toàn bộ mẫu giọng của 6 nhân vật đã được nạp sẵn sàng!")

In [ ]:
# 5. Tạo Bridge Server kết nối web vox-persona với GPT-SoVITS
%%writefile /content/GPT-SoVITS/bridge_server.py
import os, requests
from fastapi import FastAPI, Query, HTTPException
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Vox Imperium GPT-SoVITS Bridge")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

PERSONAS = {
    "jobs": {
        "ref": "/content/GPT-SoVITS/refs/jobs_speech.mp3",
        "text": "Your time is limited, so don't waste it living someone else's life.",
        "lang": "en"
    },
    "trump": {
        "ref": "/content/GPT-SoVITS/refs/trump_speech.mp3",
        "text": "Look, nobody understands deals better than me, believe me.",
        "lang": "en"
    },
    "musk": {
        "ref": "/content/GPT-SoVITS/refs/musk_speech.wav",
        "text": "From a physics first principles perspective, everything is just energy conversion.",
        "lang": "en"
    },
    "zuck": {
        "ref": "/content/GPT-SoVITS/refs/zuck_speech.mp3",
        "text": "Move fast and break things, open ecosystems always win.",
        "lang": "en"
    },
    "tesla": {
        "ref": "/content/GPT-SoVITS/refs/tesla_speech.mp3",
        "text": "If you only knew the magnificence of the 3, 6, and 9, you would have a key to the universe.",
        "lang": "en"
    },
    "xijinping": {
        "ref": "/content/GPT-SoVITS/refs/xijinping_speech.ogg",
        "text": "你好。历史的长河奔腾向前。",
        "lang": "zh"
    }
}

@app.get("/health")
def health():
    return {"status": "live", "engine": "GPT-SoVITS"}

@app.get("/synthesize")
def synthesize(persona: str = Query("musk"), text: str = Query(...)):
    p = persona.lower().strip()
    if p not in PERSONAS:
        p = "musk"
    cfg = PERSONAS[p]

    payload = {
        "text": text,
        "text_lang": "zh" if p == "xijinping" else "en",
        "ref_audio_path": cfg["ref"],
        "prompt_text": cfg["text"],
        "prompt_lang": cfg["lang"],
        "streaming_mode": False
    }

    try:
        r = requests.post("http://127.0.0.1:9880/tts", json=payload, timeout=30)
        if r.status_code == 200:
            return Response(content=r.content, media_type="audio/wav")
        else:
            raise HTTPException(status_code=r.status_code, detail=r.text)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# 6. Khởi chạy GPT-SoVITS API V2 & Mở đường hầm Cloudflare Tunnel
import subprocess, time
from pycloudflared import try_cloudflare

print("1. Đang khởi động lõi AI GPT-SoVITS API V2 trên GPU...")
api_proc = subprocess.Popen([
    "bash", "-c",
    "source activate GPTSoVITS && python api_v2.py -a 127.0.0.1 -p 9880 -c GPT_SoVITS/configs/tts_infer.yaml"
], cwd="/content/GPT-SoVITS")

time.sleep(12)

print("2. Đang khởi động Bridge Server kết nối web...")
bridge_proc = subprocess.Popen([
    "bash", "-c",
    "source activate GPTSoVITS && python bridge_server.py"
], cwd="/content/GPT-SoVITS")

time.sleep(5)

print("3. Mở kết nối Cloudflare Tunnel an toàn...")
public_url = try_cloudflare(port=8000)

print("\n" + "="*70)
print("🎉 ĐƯỜNG LINK API GPT-SoVITS CỦA BẠN ĐÃ HOÀN TẤT:")
print(f"👉 {public_url.tunnel}")
print("="*70)
print("Copy đường link 'https://xxx.trycloudflare.com' trên và dán vào ô GPT-SoVITS trên web vox-persona!")